In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler

# Cargar los archivos de entrenamiento y prueba
df_train = pd.read_csv('hoteles-entrena.csv')
df_pred = pd.read_csv('hoteles-prueba.csv')

# Preprocesamiento del archivo de entrenamiento

# Convertir 'arrival_date' a datetime y extraer características
df_train['arrival_date'] = pd.to_datetime(df_train['arrival_date'], format='%Y-%m-%d')
df_train['arrival_year'] = df_train['arrival_date'].dt.year
df_train['arrival_month'] = df_train['arrival_date'].dt.month
df_train['arrival_day'] = df_train['arrival_date'].dt.day
df_train['day_of_year'] = df_train['arrival_date'].dt.dayofyear
df_train['arrival_month_sin'] = np.sin(2 * np.pi * df_train['arrival_month'] / 12)
df_train['arrival_month_cos'] = np.cos(2 * np.pi * df_train['arrival_month'] / 12)
df_train['day_of_year_sin'] = np.sin(2 * np.pi * df_train['day_of_year'] / 365)
df_train['day_of_year_cos'] = np.cos(2 * np.pi * df_train['day_of_year'] / 365)

# Crear la columna 'stay_days' como tipo 'object' y asignar valores
df_train['stay_days'] = pd.Series('both', index=df_train.index, dtype='object')
df_train.loc[(df_train['stays_in_weekend_nights'] > 0) & (df_train['stays_in_week_nights'] == 0), 'stay_days'] = 'weekend'
df_train.loc[(df_train['stays_in_weekend_nights'] == 0) & (df_train['stays_in_week_nights'] > 0), 'stay_days'] = 'weekdays'

# Crear columnas adicionales basadas en noches
df_train['total_nights'] = df_train['stays_in_weekend_nights'] + df_train['stays_in_week_nights']
df_train['stays_in_weekend'] = (df_train['stays_in_weekend_nights'] > 0).astype(int)
df_train['weekday'] = df_train['arrival_date'].dt.weekday

# Llenar valores faltantes en columnas
df_train['country'] = df_train['country'].fillna('NON')
df_train['agent'] = df_train['agent'].fillna(0)
df_train['company'] = df_train['company'].fillna(0)

# Convertir 'children' a binario
df_train['children'] = df_train['children'].map({'children': 1, 'none': 0})

# Normalizar columnas con Min-Max Scaling
scaler = MinMaxScaler()
cols_min_max = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 
                'adults', 'previous_cancellations', 'previous_bookings_not_canceled', 
                'booking_changes', 'days_in_waiting_list', 'average_daily_rate', 'total_nights']
df_train[cols_min_max] = scaler.fit_transform(df_train[cols_min_max])

# Mapear valores binarios
df_train['hotel'] = df_train['hotel'].map({'Resort_Hotel': 1, 'City_Hotel': 0}).astype(int)
df_train['required_car_parking_spaces'] = df_train['required_car_parking_spaces'].map({'parking': 1, 'none': 0}).astype(int)

# Eliminar 'arrival_date'
df_train.drop(columns=['arrival_date'], inplace=True)

# Definir función para Target Encoding con K-Fold y manejo de NaN
def target_encoding_kfold_inplace(data, column, target, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    global_mean = data[target].mean()
    
    for train_index, val_index in kf.split(data):
        train_fold, val_fold = data.iloc[train_index], data.iloc[val_index]
        mean_target = train_fold.groupby(column)[target].mean()
        
        # Asignar valores mapeados y rellenar NaN con el promedio global
        data.loc[val_index, column] = val_fold[column].map(mean_target).fillna(global_mean)

# Aplicar Target Encoding in-place a cada columna categórica relevante
cols_target_encoding = ['meal', 'country', 'market_segment', 'distribution_channel', 
                        'reserved_room_type', 'assigned_room_type', 'deposit_type', 
                        'agent', 'company', 'customer_type', 'stay_days']
for col in cols_target_encoding:
    target_encoding_kfold_inplace(df_train, col, 'children')

# Exportar el archivo de entrenamiento limpio y normalizado
df_train.to_csv('hoteles-entrena-limpio-normalizado.csv', index=False)

# Preparación del archivo de prueba

# Convertir 'arrival_date' a datetime y extraer características
df_pred['arrival_date'] = pd.to_datetime(df_pred['arrival_date'], format='%Y-%m-%d')
df_pred['arrival_year'] = df_pred['arrival_date'].dt.year
df_pred['arrival_month'] = df_pred['arrival_date'].dt.month
df_pred['arrival_day'] = df_pred['arrival_date'].dt.day
df_pred['day_of_year'] = df_pred['arrival_date'].dt.dayofyear
df_pred['arrival_month_sin'] = np.sin(2 * np.pi * df_pred['arrival_month'] / 12)
df_pred['arrival_month_cos'] = np.cos(2 * np.pi * df_pred['arrival_month'] / 12)
df_pred['day_of_year_sin'] = np.sin(2 * np.pi * df_pred['day_of_year'] / 365)
df_pred['day_of_year_cos'] = np.cos(2 * np.pi * df_pred['day_of_year'] / 365)

# Crear 'stay_days' como tipo 'object' y asignar valores
df_pred['stay_days'] = pd.Series('both', index=df_pred.index, dtype='object')
df_pred.loc[(df_pred['stays_in_weekend_nights'] > 0) & (df_pred['stays_in_week_nights'] == 0), 'stay_days'] = 'weekend'
df_pred.loc[(df_pred['stays_in_weekend_nights'] == 0) & (df_pred['stays_in_week_nights'] > 0), 'stay_days'] = 'weekdays'

# Crear columnas adicionales
df_pred['total_nights'] = df_pred['stays_in_weekend_nights'] + df_pred['stays_in_week_nights']
df_pred['stays_in_weekend'] = (df_pred['stays_in_weekend_nights'] > 0).astype(int)
df_pred['weekday'] = df_pred['arrival_date'].dt.weekday

# Llenar valores faltantes
df_pred['country'] = df_pred['country'].fillna('NON')
df_pred['agent'] = df_pred['agent'].fillna(0)
df_pred['company'] = df_pred['company'].fillna(0)

# Normalizar las mismas columnas usando los parámetros ajustados en el entrenamiento
df_pred[cols_min_max] = scaler.transform(df_pred[cols_min_max])

# Mapear valores binarios
df_pred['hotel'] = df_pred['hotel'].map({'Resort_Hotel': 1, 'City_Hotel': 0}).astype(int)
df_pred['required_car_parking_spaces'] = df_pred['required_car_parking_spaces'].map({'parking': 1, 'none': 0}).astype(int)

# Eliminar 'arrival_date'
df_pred.drop(columns=['arrival_date'], inplace=True)

# Aplicar el target encoding calculado del entrenamiento
encoding_maps = {col: df_train.groupby(col)['children'].mean() for col in cols_target_encoding}
global_mean = df_train['children'].mean()
for col in cols_target_encoding:
    df_pred[col] = df_pred[col].map(encoding_maps[col]).fillna(global_mean)

# Exportar el archivo de prueba limpio y normalizado
df_pred.to_csv('hoteles-prueba-limpio-normalizado.csv', index=False)